# E-Commerce Order Intelligence

#### Import libraries and datasets

In [1]:
import numpy as np 
import pandas as pd 

In [2]:
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")

#### Merging orders and customers dataset


In [3]:
orders_customers = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="inner"
)

#### Merging all the tables 

In [4]:
master_df = pd.merge(
    orders_customers,
    order_items,
    on="order_id",
    how="inner"
)

In [5]:
master_df = pd.merge(
    master_df,
    products,
    on="product_id",
    how="inner"
)

#### Total Revenue Per Customer

In [7]:
revenue_per_customer = master_df.groupby(
    "customer_id"
)["price"].sum()
revenue_per_customer.head()

customer_id
00012a2ce6f8dcda20d059ce98491703     89.80
000161a058600d5901f007fab4c27140     54.90
0001fd6190edaaf884bcaf3d49edf079    179.99
0002414f95344307404f0ace7a26f1d5    149.90
000379cdec625522490c315e70c7a9fb     93.00
Name: price, dtype: float64

#### Customers with only one order

In [8]:
order_counts = orders.groupby(
    "customer_id"
)["order_id"].count()

In [9]:
single_order_customers = order_counts[
    order_counts == 1
]

In [26]:
master_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm', 'Month',
       'Year'],
      dtype='object')

#### Monthly Sales By Product

In [19]:
master_df["Month"] = pd.to_datetime(
    master_df["order_purchase_timestamp"]
).dt.to_period("M").astype(str)

monthly_product_sales = (
    master_df.groupby(["product_id", "Month"], as_index=False)["price"]
    .sum()
    .rename(columns={"price": "Sales"})
)

monthly_product_sales.head()

,product_id,Month,Sales
0,00066f42aeeb9f3007548bb9d3f33c38,2018-05,101.65
1,00088930e925c41fd95ebfe695fd2655,2017-12,129.90
2,0009406fd7479715e4bef61dd91f2462,2017-12,229.00
3,000b8f95fcb9e0096488278317764d19,2018-08,117.80
4,000d9be29b5207b54e86aa1b1ac54872,2018-04,199.00


#### Pivoting Back

In [20]:
monthly_product_sales.columns

Index(['product_id', 'Month', 'Sales'], dtype='object')

In [21]:
pivoted = monthly_product_sales.pivot_table(
    index="product_id",
    columns="Month",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

pivoted.head()

Month,2016-09,2016-10,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,...,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09
product_id,,,,,,,,,,,,,,,,,,,,,
00066f42aeeb9f3007548bb9d3f33c38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,101.65,0.0,0.0,0.0,0.0
00088930e925c41fd95ebfe695fd2655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,129.9,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0
0009406fd7479715e4bef61dd91f2462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,229.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0
000b8f95fcb9e0096488278317764d19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,117.8,0.0
000d9be29b5207b54e86aa1b1ac54872,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,199.0,0.00,0.0,0.0,0.0,0.0


#### Extract Year and Month

In [22]:
master_df["order_purchase_timestamp"] = pd.to_datetime(
    master_df["order_purchase_timestamp"]
)

In [23]:
master_df["Year"] = master_df[
    "order_purchase_timestamp"
].dt.year

In [24]:
master_df["Month"] = master_df[
    "order_purchase_timestamp"
].dt.month

In [28]:
master_df["customer_city"] = master_df[
    "customer_city"
].str.upper()

#### Bonus Challenge

In [30]:
master_df["delivered"] = pd.to_datetime(
    master_df["order_delivered_customer_date"]
)

master_df["purchase"] = pd.to_datetime(
    master_df["order_purchase_timestamp"]
)

In [31]:
master_df["delivery_days"] = (
    master_df["delivered"] -
    master_df["purchase"]
).dt.days

In [32]:
master_df["delivery_days"].mean()

12.007722603361284

In [33]:
master_df.groupby("customer_city")[
    "delivery_days"
].mean().sort_values(
    ascending=False
)

customer_city
NOVO BRASIL                 148.00
CAPINZAL DO NORTE           109.00
ADHEMAR DE BARROS            97.00
SANTA CRUZ DE GOIAS          86.00
ARACE                        85.75
                             ...  
PORTO DOS GAUCHOS              NaN
SANTO ANTONIO DE GOIAS         NaN
SAO FERNANDO                   NaN
SAO FRANCISCO DO HUMAITA       NaN
SAO JOAO DO ITAPERIU           NaN
Name: delivery_days, Length: 4110, dtype: float64